In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

/Users/bohdana.ivakhnenko/.pyenv/versions/3.13.0/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
model_name = "openai-community/gpt2"
device = torch.device("cuda" if torch.cuda.is_available() else "mps")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
   # quantization_config=quantization_config,
    # device_map="auto"
).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [32]:
inputs = tokenizer("Once upon a time, there was a magical forest", return_tensors="pt").to(model.device)
print("inputs", tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False))
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


inputs Once upon a time, there was a magical forest
Once upon a time, there was a magical forest, and the forest was filled with the magic of the gods. The gods were the gods of the forest, and the forest was filled with the magic of the gods. The gods were the gods of the forest, and the forest was filled with the magic of the gods. The gods were the gods of the forest, and the forest was filled with the magic of the gods. The gods were the gods of the forest, and the forest was filled with the magic of the gods. The gods were


In [40]:
type(model)

transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel

In [39]:
inputs = tokenizer("Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950", return_tensors="pt").to(model.device)
print("inputs", inputs["input_ids"].shape)
# print("inputs", tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False))
model.train()
outputs = model(**inputs, labels=inputs["input_ids"], max_new_tokens=100)
print("outputs", outputs)

inputs torch.Size([1, 87])


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


outputs CausalLMOutputWithCrossAttentions(loss=tensor(4.8780, device='mps:0', grad_fn=<NllLossBackward0>), logits=tensor([[[ -37.3806,  -36.9769,  -38.0924,  ...,  -45.8806,  -43.2459,
           -36.1123],
         [ -53.8253,  -51.3927,  -56.5733,  ...,  -59.0923,  -59.5924,
           -53.1848],
         [ -82.3190,  -81.2187,  -84.2968,  ...,  -89.9397,  -86.6560,
           -81.9845],
         ...,
         [-100.4777,  -98.8192,  -99.2554,  ..., -106.2129, -106.0291,
           -93.5408],
         [ -67.5467,  -67.5232,  -67.1656,  ...,  -77.9705,  -74.6140,
           -67.7045],
         [ -86.4240,  -85.4851,  -86.0185,  ...,  -96.2117,  -94.4915,
           -83.0853]]], device='mps:0', grad_fn=<LinearBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)


TypeError: argument 'ids': 'float' object cannot be converted to 'Sequence'

In [41]:
tokenizer.decode([50256])

'<|endoftext|>'

In [42]:
tokenizer.decode([tokenizer.eos_token])

TypeError: argument 'ids': 'str' object cannot be interpreted as an integer

In [43]:
tokenizer.eos_token

'<|endoftext|>'